# Leave-One-Gene-Out Analysis

In this analysis, we perform a leave-one-gene-out (LOGO) evaluation to assess whether data leakage from pooling single-cell profiles inflates phenotypic activity scores.

For each gene known to be associated with a target phenotype (e.g., **Prometaphase**):
1. Its associated cells are **excluded** from building the on/off signatures.
2. The on/off signatures are computed from the remaining cells in the target phenotype against the negative controls.
3. The **excluded gene's cells** are then scored against those signatures.

Here, **the phenotype (e.g., Prometaphase) is set as the target** and **negative controls (negcon) are used as the reference baseline**. The scores reflect how close the held-out gene's cells are to the target phenotype. This means:
- **Lower scores = good** — the held-out gene's cells are morphologically similar to the target phenotype, indicating a genuine phenotypic signal.
- If data leakage were present (i.e., the gene's own cells contributed to the signature), scores would be artificially low. Under the LOGO design, **scores that remain low confirm the signal is real** — those cells genuinely resemble the target phenotype even when they played no role in building the signature.

To create a null distribution or negative control baseline, we shuffle the feature profiles to break the biological relationships while preserving the data structure.

## Importing packages

In [1]:
import sys
import pathlib
import polars as pl

import numpy as np
from tqdm import tqdm

sys.path.append("../../")
from utils.io_utils import load_profiles, load_configs
from buscar.signatures import get_signatures
from buscar.metrics import calculate_buscar_scores
from utils.data_utils import shuffle_feature_profiles

## Setting helper functions

In [2]:
def shuffle_signatures(
    on_sig: list[str], off_sig: list[str], all_features: list[str], seed: int = 0
) -> tuple[list[str], list[str]]:
    """
    Breaks biological meaning of on/off signatures by randomly sampling
    features from the full feature space, while preserving the original
    on/off size ratio.

    Preserves:
      - len(on_sig) and len(off_sig)  ← ratio intact
      - Features drawn from same pool as real signatures

    Breaks:
      - Which specific features are "on" vs "off"
      - Any biological grouping derived from KS test
    """
    rng = np.random.default_rng(seed)

    n_on = len(on_sig)
    n_off = len(off_sig)

    # guard: need enough features to fill both without overlap
    if n_on + n_off > len(all_features):
        raise ValueError(
            f"Not enough features ({len(all_features)}) to fill "
            f"on ({n_on}) + off ({n_off}) without replacement"
        )

    # sample without replacement so on and off don't overlap
    sampled = rng.choice(all_features, size=n_on + n_off, replace=False)

    shuffled_on = sampled[:n_on].tolist()
    shuffled_off = sampled[n_on:].tolist()

    return shuffled_on, shuffled_off

## setting input and output paths

In [3]:
# set data path
data_dir = pathlib.Path("../0.download-data/data/sc-profiles/").resolve(strict=True)
mitocheck_data = (data_dir / "mitocheck").resolve(strict=True)

# sertting mitocheck paths
mitocheck_profile_path = (mitocheck_data / "mitocheck_concat_profiles.parquet").resolve(
    strict=True
)

# setting config paths
# ensg_genes_config_path = (
#     mitocheck_data / "mitocheck_ensg_to_gene_symbol_mapping.json"
# ).resolve(strict=True)
mitocheck_feature_space_config = (
    mitocheck_data / "mitocheck_feature_space_configs.json"
).resolve(strict=True)

# set results output path
results_dir = pathlib.Path("./results/").resolve()
results_dir.mkdir(exist_ok=True)

moa_analysis_output = (results_dir / "logo_analysis").resolve()
moa_analysis_output.mkdir(exist_ok=True)

## Loading data

In [4]:
# load in configs
# ensg_genes_decoder = load_configs(ensg_genes_config_path)
feature_space_configs = load_configs(mitocheck_feature_space_config)
meta_feats = feature_space_configs["metadata-features"]
morph_feats = feature_space_configs["morphology-features"]

In [5]:
# load in mitocheck profiles
mitocheck_df = load_profiles(mitocheck_profile_path)
mitocheck_df = mitocheck_df.select(pl.col(meta_feats + morph_feats))

# removing failed qc
mitocheck_df = mitocheck_df.filter(pl.col("Metadata_Gene") != "failed QC")

# replace "negative_control" and "positive_control" values in Metadata_Gene with
# "negcon" and "poscon" respectively
mitocheck_df = mitocheck_df.with_columns(
    pl.col("Metadata_Gene").map_elements(
        lambda x: (
            "negcon"
            if x == "negative control"
            else ("poscon" if x == "positive control" else x)
        ),
        return_dtype=pl.String,
    )
)

In [6]:
labeled_mitocheck_df = mitocheck_df.filter(
    (pl.col("Mitocheck_Phenotypic_Class") != "negcon")
    & (pl.col("Mitocheck_Phenotypic_Class") != "poscon")
)

print("Shape of the labeled mitocheck profiles:", labeled_mitocheck_df.shape)
labeled_mitocheck_df.head()

Shape of the labeled mitocheck profiles: (2503, 170)


Mitocheck_Phenotypic_Class,Cell_UUID,Location_Center_X,Location_Center_Y,Metadata_Plate,Metadata_Well,Metadata_Frame,Metadata_Site,Metadata_Plate_Map_Name,Metadata_DNA,Metadata_Gene,Metadata_Gene_Replicate,Metadata_treatment_type,Intensity_StdIntensityEdge_DNA,Texture_Variance_DNA_3_03_256,Granularity_7_DNA,Texture_InverseDifferenceMoment_DNA_3_02_256,RadialDistribution_FracAtD_DNA_3of4,AreaShape_MinorAxisLength,Texture_Contrast_DNA_3_01_256,Intensity_IntegratedIntensity_DNA,RadialDistribution_MeanFrac_DNA_2of4,AreaShape_Zernike_6_2,Granularity_13_DNA,Texture_Variance_DNA_3_01_256,Texture_Entropy_DNA_3_02_256,AreaShape_Area,Granularity_6_DNA,Neighbors_NumberOfNeighbors_Adjacent,Granularity_10_DNA,Intensity_StdIntensity_DNA,Texture_SumVariance_DNA_3_00_256,Neighbors_FirstClosestDistance_Adjacent,AreaShape_ConvexArea,AreaShape_Zernike_8_8,AreaShape_Zernike_9_3,AreaShape_Zernike_0_0,…,AreaShape_Zernike_1_1,AreaShape_MajorAxisLength,AreaShape_MaximumRadius,Intensity_LowerQuartileIntensity_DNA,Texture_InfoMeas2_DNA_3_00_256,AreaShape_Zernike_9_1,Texture_Entropy_DNA_3_00_256,AreaShape_Compactness,AreaShape_Zernike_5_3,RadialDistribution_FracAtD_DNA_4of4,AreaShape_Zernike_3_1,AreaShape_Perimeter,Texture_Contrast_DNA_3_00_256,AreaShape_MedianRadius,Granularity_14_DNA,AreaShape_Zernike_7_1,AreaShape_Zernike_7_3,Intensity_UpperQuartileIntensity_DNA,Texture_DifferenceVariance_DNA_3_02_256,Intensity_MeanIntensityEdge_DNA,AreaShape_Zernike_4_2,AreaShape_FormFactor,AreaShape_Zernike_2_0,AreaShape_Zernike_8_4,Neighbors_PercentTouching_Adjacent,AreaShape_Zernike_4_0,AreaShape_Zernike_8_6,AreaShape_Zernike_9_9,Granularity_15_DNA,Texture_AngularSecondMoment_DNA_3_03_256,Texture_InfoMeas1_DNA_3_03_256,Texture_SumAverage_DNA_3_02_256,Granularity_3_DNA,AreaShape_Zernike_6_4,Texture_Entropy_DNA_3_03_256,AreaShape_Zernike_5_5,AreaShape_MaxFeretDiameter
str,str,i64,i64,str,i64,i64,i64,str,str,str,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Large""","""21da27ab-873a-41f4-ab98-49170c…",397,618,"""LT0010_27""",173,83,1,"""LT0010_27_173""","""LT0010_27/LT0010_27_173_83.tif""","""RAB21""",1,"""trt""",-0.226037,-0.321772,-0.162936,-0.069676,-0.429201,1.170372,-0.283944,0.795352,-0.441865,2.060887,-0.025113,-0.322017,0.513607,2.514724,0.821603,-0.419813,-0.046809,-0.350783,-0.323359,0.757074,2.503212,-0.606715,-1.008395,-1.038763,…,-0.928531,2.813648,1.175383,0.291235,0.222914,-1.588903,0.567256,0.438754,-0.816313,0.759013,-0.33483,2.252821,-0.283365,1.643822,-0.026787,-1.160951,-0.519895,-0.180181,-0.358357,0.291481,-1.057124,-0.679105,1.180296,0.569697,-0.39273,-1.401342,1.232027,0.459925,-0.017574,-0.339021,0.587211,-0.121025,-1.022403,0.755476,0.611549,-0.621212,2.574174
"""Large""","""82f7949b-4ea2-45c8-8dd9-7854ca…",359,584,"""LT0010_27""",173,83,1,"""LT0010_27_173""","""LT0010_27/LT0010_27_173_83.tif""","""RAB21""",1,"""trt""",-0.204892,-0.242154,-0.162936,-0.471068,-0.59935,2.14545,-0.227807,1.928243,-0.593857,1.285899,-0.025113,-0.249915,0.876355,3.493695,2.16752,-0.419813,-0.046809,-0.110927,-0.245543,0.757074,3.444921,-0.563294,-0.312259,-0.402259,…,-0.853936,2.988465,2.130062,0.908971,0.371857,-0.185326,0.885164,0.044005,-1.364292,0.846376,0.088275,2.767656,-0.238553,2.263011,-0.026787,-1.071024,-0.667567,0.223784,-0.426788,0.624018,-0.84918,-0.147518,0.8452,-0.425917,-0.39273,-0.066645,-0.366503,0.233967,-0.017574,-0.360228,0.284336,0.316861,-1.782799,1.184921,0.886059,0.696551,2.870924
"""Large""","""cec7234f-fe35-4411-aded-f8112b…",383,685,"""LT0010_27""",173,83,1,"""LT0010_27_173""","""LT0010_27/LT0010_27_173_83.tif""","""RAB21""",1,"""trt""",-0.0708,-0.221919,-0.162936,-0.298418,-0.185095,1.168898,-0.204286,1.714938,-0.583743,1.505383,-0.025113,-0.218458,0.894006,2.752992,0.5105,-0.419813,-0.046809,0.001931,-0.216386,1.019491,2.781

In [7]:
# Creating a proportion dataframe for all genes and phenotypic classes
cell_proportion_df = (
    mitocheck_df.filter(
        (pl.col("Mitocheck_Phenotypic_Class") != "negcon")
        & (pl.col("Mitocheck_Phenotypic_Class") != "poscon")
    )
    .group_by(["Metadata_Gene", "Mitocheck_Phenotypic_Class"])
    .agg(pl.len().alias("count"))
    .with_columns(pl.col("count").sum().over("Metadata_Gene").alias("total_count"))
    .with_columns((pl.col("count") / pl.col("total_count")).alias("proportion"))
)

Get cell state information

In [8]:
cell_states = (
    # remove negcon and poscon since they do not have cell state information
    mitocheck_df.filter(
        (pl.col("Mitocheck_Phenotypic_Class") != "negcon")
        & (pl.col("Mitocheck_Phenotypic_Class") != "poscon")
    )
    .select("Mitocheck_Phenotypic_Class")
    .unique()
    .to_series()
    .to_list()
)

## LOGO analysis 

In [9]:
# parameters for the analysis
shuffle_flag = True
seed = 0

In [10]:
if shuffle_flag:
    print("Shuffling the mitocheck profiles...")
    shuffled_mitocheck_df = shuffle_feature_profiles(
        profiles=labeled_mitocheck_df,
        feature_cols=morph_feats,
        method="column",
        label_col="Mitocheck_Phenotypic_Class",
        seed=seed,
    )

Shuffling the mitocheck profiles...


In [11]:
# select data based on shuffle_flag
profiles = shuffled_mitocheck_df if shuffle_flag else labeled_mitocheck_df

negcon_profiles = mitocheck_df.filter(
    pl.col("Mitocheck_Phenotypic_Class") == "negcon"
).sample(fraction=0.01)

on_off_sigs = []
min_cells = 2

results_df = []
for cell_state in tqdm(cell_states, desc="Processing cell states"):
    # state of interest for this cell state
    selected_state = profiles.filter(pl.col("Mitocheck_Phenotypic_Class") == cell_state)

    # genes that are associated with this cell state
    genes_associated_with_state = (
        selected_state.select("Metadata_Gene").unique().to_series().to_list()
    )

    # genes that are not associated with this cell state
    genes_not_associated_with_state = (
        profiles.filter(~pl.col("Metadata_Gene").is_in(genes_associated_with_state))
        .select("Metadata_Gene")
        .unique()
        .to_series()
        .to_list()
    )

    associated_gene_scores = []
    for gene in tqdm(
        genes_associated_with_state,
        desc=f"  Processing genes for {cell_state}",
        leave=False,
    ):
        # filter the target profiles to only include cells treated with the current
        # gene of interest
        heldout_df = selected_state.filter(pl.col("Metadata_Gene") == gene)

        # skip genes with too few cells (EMD requires >= 2 samples)
        if heldout_df.height < min_cells:
            print(
                f"Skipping gene '{gene}': only {heldout_df.height} cell(s), need >= "
                f"{min_cells}"
            )
            # create an empty dataframe with the same structure as the
            # associated_gene_score to maintain consistency
            associated_gene_score = pl.DataFrame(
                {
                    "target": pl.Series([cell_state], dtype=pl.String),
                    "perturbation": pl.Series([gene], dtype=pl.String),
                    "on_buscar_scores": pl.Series([None], dtype=pl.Float64),
                    "off_buscar_scores": pl.Series([None], dtype=pl.Float64),
                    "is_reference_distance": pl.Series([None], dtype=pl.Boolean),
                    "proportion": pl.Series([None], dtype=pl.Float64),
                    "is_associated": pl.Series([None], dtype=pl.Boolean),
                }
            )
            associated_gene_scores.append(associated_gene_score)
            continue

        # remove the current gene's cells from the positive control pool
        # to prevent data leakage: the gene being ranked must not influence its own
        # signature
        state_pool = selected_state.filter(pl.col("Metadata_Gene") != gene)

        # generate on and off signatures (leave-one-out: current gene's cells excluded)
        morph_feats = feature_space_configs["morphology-features"]
        on_sig, off_sig, _ = get_signatures(
            state_pool,
            negcon_profiles,
            morph_feats=morph_feats,
            test_method="ks_test",
            p_threshold=0.05,
            seed=seed,
        )

        # concatenating negcon, the gene that has been held out, and the state_pool
        test_df = pl.concat([negcon_profiles, heldout_df, state_pool])

        test_df = test_df.with_columns(
            pl.when(pl.col("Metadata_Gene") == "negcon")
            .then(pl.lit("negcon"))
            .when(pl.col("Metadata_Gene") == gene)
            .then(pl.col("Metadata_Gene"))
            .otherwise(pl.lit(cell_state))  # label pooled target as cell state
            .alias("_labeled_references")
        )

        if shuffle_flag:
            # shuffle the on and off signatures and shuffle
            on_sig, off_sig = shuffle_signatures(
                on_sig, off_sig, morph_feats, seed=seed
            )
            test_df = shuffle_feature_profiles(
                profiles=test_df,
                feature_cols=morph_feats,
                method="column",
                seed=seed,
            )

        # if no signature was found, skip the gene
        if len(on_sig) == 0 or len(off_sig) == 0:
            print(f"skipping {gene}")
            continue

        # rank the gene using the generated signatures
        associated_gene_score = calculate_buscar_scores(
            profiles=test_df,
            meta_cols=feature_space_configs["metadata-features"],
            on_morphology_signature=on_sig,
            off_morphology_signature=off_sig,
            target=cell_state,
            ref_state="negcon",
            perturbation_col="_labeled_references",
            n_threads=1,
            raw_emd_scores=False,
        )

        # calculate the proportion of cells that make up this phenotype with the
        # current gene perturbation
        try:
            cell_state_proportion = cell_proportion_df.filter(
                (pl.col("Metadata_Gene") == gene)
                & (pl.col("Mitocheck_Phenotypic_Class") == cell_state)
            )["proportion"][0]
        except IndexError:
            cell_state_proportion = 0.0

        # remove negcon scores; we are only interested in the scores of the gene
        associated_gene_score = associated_gene_score.filter(
            pl.col("perturbation") != "negcon"
        )

        # add cell state proportion to the associated gene scores df
        associated_gene_score = associated_gene_score.with_columns(
            pl.lit(cell_state_proportion).alias("proportion"),
            pl.lit(True).alias("is_associated"),
        )

        # store on and off signatures
        on_off_sigs.append((cell_state, on_sig, off_sig))
        associated_gene_scores.append(associated_gene_score)

    if len(associated_gene_scores) > 0:
        associated_gene_scores = pl.concat(associated_gene_scores)
    else:
        associated_gene_scores = pl.DataFrame(
            schema={
                "target": pl.String,
                "perturbation": pl.String,
                "on_buscar_scores": pl.Float64,
                "off_buscar_scores": pl.Float64,
                "is_reference_distance": pl.Boolean,
                "proportion": pl.Float64,
                "is_associated": pl.Boolean,
            }
        )

    # Step 2: rank genes that are not associated with this cell state

    # create on and off sigs with pooled poscon cell state
    on_sig, off_sig, _ = get_signatures(
        ref_profiles=selected_state,
        target_profiles=negcon_profiles,
        morph_feats=morph_feats,
        test_method="ks_test",
        p_threshold=0.05,
        seed=seed,
    )

    test_non_associated_df = pl.concat(
        [
            selected_state,
            profiles.filter(
                pl.col("Metadata_Gene").is_in(genes_not_associated_with_state)
            ),
            negcon_profiles,
        ]
    )

    # the genes not associated with the cell state, and the target phenotype pool
    test_non_associated_df = test_non_associated_df.with_columns(
        pl.when(pl.col("Metadata_Gene") == "negcon")
        .then(pl.lit("negcon"))
        .when(pl.col("Metadata_Gene").is_in(genes_associated_with_state))
        .then(pl.lit(cell_state))  # label pooled target as cell state
        .otherwise(pl.col("Metadata_Gene"))  # keep non-associated as gene names
        .alias("_labeled_references")
    )

    if shuffle_flag:
        on_sig, off_sig = shuffle_signatures(on_sig, off_sig, morph_feats, seed=seed)
        test_non_associated_df = shuffle_feature_profiles(
            profiles=test_non_associated_df,
            feature_cols=morph_feats,
            method="column",
            seed=seed,
        )

    # rank all treatments that are not associated with this cell state using the pooled
    # poscon signatures
    not_associated_gene_scores = calculate_buscar_scores(
        profiles=test_non_associated_df,
        meta_cols=meta_feats,
        on_morphology_signature=on_sig,
        off_morphology_signature=off_sig,
        target=cell_state,
        ref_state="negcon",
        perturbation_col="_labeled_references",
        n_threads=1,
        seed=seed,
        raw_emd_scores=False,
    )

    # remove scores of genes that are associated with the cell state
    not_associated_gene_scores = not_associated_gene_scores.filter(
        pl.col("perturbation").is_in(genes_not_associated_with_state)
    )

    # add proportion of cells; if a gene has no cells in this state, assign 0
    not_associated_gene_scores = not_associated_gene_scores.join(
        cell_proportion_df.select(
            ["Metadata_Gene", "Mitocheck_Phenotypic_Class", "proportion"]
        ),
        left_on=["perturbation", "target"],
        right_on=["Metadata_Gene", "Mitocheck_Phenotypic_Class"],
        how="left",
    ).with_columns(
        pl.col("proportion").fill_null(0.0), pl.lit(False).alias("is_associated")
    )

    # final result for this cell state
    results_df.append(
        pl.concat([associated_gene_scores, not_associated_gene_scores], how="vertical")
    )

# step 3: store results
if len(results_df) > 0:
    results_df = pl.concat(results_df)
    output_filename = f"{'shuffled' if shuffle_flag else 'original'}_mitocheck_logo_analysis_results.parquet"
    results_df.write_parquet(moa_analysis_output / output_filename)
else:
    print("No results generated to save.")

Processing cell states:   0%|          | 0/16 [00:00<?, ?it/s]

Skipping gene 'OGG1': only 1 cell(s), need >= 2


Skipping gene 'ENSG00000110675': only 1 cell(s), need >= 2


Skipping gene 'ENSG00000175216': only 1 cell(s), need >= 2


Skipping gene 'RGR': only 1 cell(s), need >= 2


...

Skipping gene 'ENSG00000148826': only 1 cell(s), need >= 2


Processing cell states:  94%|█████████▍| 15/16 [04:13<00:11, 11.20s/it]

Skipping gene 'ENSG00000116641': only 1 cell(s), need >= 2


Skipping gene 'PRC1': only 1 cell(s), need >= 2


Processing cell states: 100%|██████████| 16/16 [05:20<00:00, 20.01s/it] 1.50s/it]
